# From Chain Rule to Generated Text: a Tiny Bayesian Language Model

Notebook for the *Bayesian Networks and Autoregressive Language Models* lab. Both the
first-order and second-order count-based models below were implemented with help from an
LLM (Claude), inspected, and tested against the normalisation property the lab specifies.

## Part I/II - Question 1: why decompose the joint probability?

$$P(X_1,\dots,X_T) = P(X_1)\prod_{t=2}^{T} P(X_t \mid X_1,\dots,X_{t-1})$$

This chain-rule factorisation is useful for *generation* because it turns one hard problem
(producing an entire sentence at once) into a sequence of easy ones (producing one token,
given everything already written). A generator can therefore work left to right: sample
$X_1$, then sample $X_2$ conditioned on $X_1$, then $X_3$ conditioned on $X_1,X_2$, and so
on - which is exactly how the sampler built later in this notebook operates.

## Question 2: the independence assumption of the simple network

The chain $X_1 \to X_2 \to \cdots \to X_T$ factorises the joint as

$$P(X_1,\dots,X_T) = P(X_1)\prod_{t=2}^{T} P(X_t \mid X_{t-1}),$$

which assumes $P(X_t \mid X_1,\dots,X_{t-1}) \approx P(X_t \mid X_{t-1})$ for every $t$: each
token depends only on the *immediately preceding* one, and is conditionally independent of
everything earlier given that one token. This is the first-order Markov assumption.

## Part III - building a small dataset

In [1]:
RAW_SENTENCES = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

START, END = "<START>", "<END>"

def tokenize(sentence):
    return [START] + sentence.lower().split() + [END]

CORPUS = [tokenize(s) for s in RAW_SENTENCES]
for toks in CORPUS:
    print(toks)

['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'dog', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'cat', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '<END>']


## Part IV - Question 3: the first-order conditional probability table

$$P(w_j \mid w_i) = \frac{C(w_i, w_j)}{\sum_k C(w_i, w_k)}$$

built directly from bigram counts over the tokenised corpus.

In [2]:
from collections import defaultdict, Counter

def bigram_counts(corpus):
    counts = defaultdict(Counter)
    for toks in corpus:
        for a, b in zip(toks, toks[1:]):
            counts[a][b] += 1
    return counts

def to_probabilities(counts):
    probs = {}
    for w, nxt_counts in counts.items():
        total = sum(nxt_counts.values())
        probs[w] = {w2: c / total for w2, c in nxt_counts.items()}
    return probs

COUNTS_1 = bigram_counts(CORPUS)
PROBS_1 = to_probabilities(COUNTS_1)

for w in ["the", "cat", "dog", "sat", "ran"]:
    print(w, "->", PROBS_1.get(w, {}))

the -> {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}
cat -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
dog -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
sat -> {'on': 1.0}
ran -> {'to': 1.0}


In [3]:
VOCAB = sorted({tok for toks in CORPUS for tok in toks})
print("Zero-probability transitions for the, cat, dog, sat, ran:")
for w in ["the", "cat", "dog", "sat", "ran"]:
    seen = PROBS_1.get(w, {})
    zero = [v for v in VOCAB if v not in seen]
    print(f"  {w}: {zero}")

Zero-probability transitions for the, cat, dog, sat, ran:
  the: ['<END>', '<START>', 'on', 'ran', 'sat', 'the', 'to']
  cat: ['<END>', '<START>', 'cat', 'dog', 'mat', 'on', 'park', 'rug', 'the', 'to']
  dog: ['<END>', '<START>', 'cat', 'dog', 'mat', 'on', 'park', 'rug', 'the', 'to']
  sat: ['<END>', '<START>', 'cat', 'dog', 'mat', 'park', 'ran', 'rug', 'sat', 'the', 'to']
  ran: ['<END>', '<START>', 'cat', 'dog', 'mat', 'on', 'park', 'ran', 'rug', 'sat', 'the']


`sat` and `ran` only ever observed `on`/`to` follow them in this tiny corpus, so every
other vocabulary word (including `<END>` directly, or `the` itself) has zero probability
after them - the table only knows what it has actually seen.

## Part V - asking an LLM for the first-order implementation

Prompt used (the lab's suggested one, adapted to the dataset above):

> *Write a simple Python implementation of a first-order autoregressive language model that:
> takes tokenised sentences as training data; counts transitions between consecutive tokens;
> builds* P(X_t | X_{t-1}) *from those counts; can display the distribution for a given
> previous token; predicts the single most probable next token; generates a sentence by
> repeatedly sampling until* `<END>` *is produced. Do not use any machine-learning library.*

The functions above (`bigram_counts`, `to_probabilities`) already cover the counting and CPT
construction; prediction and sampling are added next.

In [4]:
import random

def most_likely_next(word, probs=PROBS_1):
    dist = probs.get(word, {})
    if not dist:
        return None
    return max(dist, key=dist.get)

def sample_next(word, probs=PROBS_1, rng=random):
    dist = probs.get(word, {})
    if not dist:
        return END
    words, weights = zip(*dist.items())
    return rng.choices(words, weights=weights, k=1)[0]

## Part VI - Questions 4-7: reading the generated code before running it

**Q4. Where are the transition counts stored?** In `COUNTS_1`, a dictionary mapping each
word to a `Counter` of the words observed immediately after it.

**Q5. Where is $P(X_t \mid X_{t-1})$ computed?** In `to_probabilities`, which divides each
count by the total count of transitions out of that word.

**Q6. How does the program choose the next word - greedy or sampling?** Both are
implemented separately: `most_likely_next` always takes the arg-max (greedy), while
`sample_next` draws from the full distribution with `random.choices` weighted by probability
(sampling). The difference is that greedy always gives the same output for a given input,
while sampling can give different output for the same input on different runs, in proportion
to how likely each continuation is.

**Q7. What happens on an unseen word?** `probs.get(word, {})` returns an empty dictionary,
and both helper functions treat that as "nothing to predict" - `most_likely_next` returns
`None` and `sample_next` falls back to `<END>`, rather than crashing.

## Part VII - Question 8: testing that probabilities actually normalise

$$\sum_v P(v \mid w) = 1 \quad \text{for every } w$$

In [5]:
def check_normalisation(probs):
    for context, dist in probs.items():
        total = sum(dist.values())
        print(f"{str(context):>20}: total = {total:.6f}")
        assert abs(total - 1.0) < 1e-9, f"{context} does not normalise to 1"

check_normalisation(PROBS_1)
print("\nAll rows sum to 1.0, as required.")

             <START>: total = 1.000000
                 the: total = 1.000000
                 cat: total = 1.000000
                 sat: total = 1.000000
                  on: total = 1.000000
                 mat: total = 1.000000
                 rug: total = 1.000000
                 dog: total = 1.000000
                 ran: total = 1.000000
                  to: total = 1.000000
                park: total = 1.000000

All rows sum to 1.0, as required.


If one of these totals had come out as, say, 0.87, that would mean the implementation
is *not* correctly normalising the counts for that word - some probability mass is missing,
which could happen from a counting bug (e.g. not counting every bigram) or a division
mistake, and the model would no longer be a valid probability distribution even if it still
produced output that looked plausible.

## Part VIII - Question 9: predicting the next word for several contexts

In [6]:
for w in ["the", "cat", "dog", "sat", "ran"]:
    dist = PROBS_1.get(w, {})
    print(f"P(next | {w}) = {dist}")
    print(f"   arg max -> {most_likely_next(w)}")

P(next | the) = {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}
   arg max -> cat
P(next | cat) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
   arg max -> sat
P(next | dog) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
   arg max -> sat
P(next | sat) = {'on': 1.0}
   arg max -> on
P(next | ran) = {'to': 1.0}
   arg max -> to


The most probable continuation is not always what a human would guess first. For
example after `the`, both `cat` and `dog` happen to have been seen with roughly similar
frequency in this tiny corpus, so the arg-max can be sensitive to exactly how many of each
sentence were included - a human's sense of "the most natural word" draws on a lifetime of
language exposure, while this model only knows the six sentences it was given. That gap is
precisely the difference between *a probability model fit to a specific sample* and *human
linguistic expectations built from vastly more experience*.

## Part IX - generating new sentences by repeated sampling

In [7]:
def generate(probs=PROBS_1, max_len=20, rng=random):
    tokens = [START]
    while tokens[-1] != END and len(tokens) < max_len:
        tokens.append(sample_next(tokens[-1], probs, rng))
    return " ".join(tokens[1:-1]) if tokens[-1] == END else " ".join(tokens[1:])

random.seed(7)
generated = [generate() for _ in range(20)]
for s in generated:
    print(s)

with open("generated_sentences.txt", "w") as f:
    f.write("\n".join(generated) + "\n")
print("\nsaved to generated_sentences.txt")

the cat sat on the mat
the cat sat on the rug
the cat sat on the mat
the park
the cat sat on the rug
the rug
the cat ran to the dog sat on the dog sat on the park
the park
the dog sat on the dog ran to the mat
the rug
the park
the cat ran to the dog sat on the cat sat on the cat ran to the mat
the rug
the dog ran to the mat
the cat sat on the rug
the cat sat on the park
the dog ran to the dog ran to the mat
the cat sat on the mat
the cat sat on the park
the mat

saved to generated_sentences.txt


## Part X - Question 10: greedy vs. sampling generation

In [8]:
def generate_greedy(probs=PROBS_1, max_len=20):
    tokens = [START]
    while tokens[-1] != END and len(tokens) < max_len:
        nxt = most_likely_next(tokens[-1], probs)
        if nxt is None:
            break
        tokens.append(nxt)
    return " ".join(t for t in tokens if t not in (START, END))

print("Greedy (mode A), run 5 times:")
for _ in range(5):
    print(" ", generate_greedy())

random.seed(3)
print("\nSampling (mode B), 5 sentences:")
for _ in range(5):
    print(" ", generate())

Greedy (mode A), run 5 times:
  the cat sat on the cat sat on the cat sat on the cat sat on the cat sat
  the cat sat on the cat sat on the cat sat on the cat sat on the cat sat
  the cat sat on the cat sat on the cat sat on the cat sat on the cat sat
  the cat sat on the cat sat on the cat sat on the cat sat on the cat sat
  the cat sat on the cat sat on the cat sat on the cat sat on the cat sat

Sampling (mode B), 5 sentences:
  the rug
  the dog sat on the mat
  the rug
  the dog sat on the rug
  the cat ran to the cat ran to the park


Greedy generation produces the *same* sentence every time, since it always follows the
single highest-probability branch from `<START>`. Sampling produces visibly different
sentences across runs, because at each step it can take any transition with nonzero
probability, weighted by how often it was seen. Mode B is therefore the one that actually
produces variation; mode A is deterministic given the trained probabilities.

## Part XI/XII - a second-order model

Instead of $P(X_t \mid X_{t-1})$, the second-order model estimates
$P(X_t \mid X_{t-2}, X_{t-1})$, with graph structure $X_{t-2} \to X_t \leftarrow X_{t-1}$.
Prompt used to extend the code:

> *Modify the first-order model into a second-order model that estimates*
> P(X_t | X_{t-2}, X_{t-1}) *using counts of observed (previous-pair, next-token) triples.
> Do not replace the counting approach with a neural network.*

In [9]:
def trigram_counts(corpus):
    counts = defaultdict(Counter)
    for toks in corpus:
        for a, b, c in zip(toks, toks[1:], toks[2:]):
            counts[(a, b)][c] += 1
    return counts

COUNTS_2 = trigram_counts(CORPUS)
PROBS_2 = to_probabilities(COUNTS_2)

for ctx in [(START, "the"), ("the", "cat"), ("the", "dog")]:
    print(ctx, "->", PROBS_2.get(ctx, {}))

check_normalisation(PROBS_2)

('<START>', 'the') -> {'cat': 0.5, 'dog': 0.5}
('the', 'cat') -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
('the', 'dog') -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
  ('<START>', 'the'): total = 1.000000
      ('the', 'cat'): total = 1.000000
      ('cat', 'sat'): total = 1.000000
       ('sat', 'on'): total = 1.000000
       ('on', 'the'): total = 1.000000
      ('the', 'mat'): total = 1.000000
      ('the', 'rug'): total = 1.000000
      ('the', 'dog'): total = 1.000000
      ('dog', 'sat'): total = 1.000000
      ('dog', 'ran'): total = 1.000000
       ('ran', 'to'): total = 1.000000
       ('to', 'the'): total = 1.000000
     ('the', 'park'): total = 1.000000
      ('cat', 'ran'): total = 1.000000


Every stored context is a *pair* of preceding tokens, e.g. `(START, "the")`, so the very
first word of a sentence still has to come from somewhere: there is no two-token context
before anything has been generated yet. The generator below bootstraps the first word with
the first-order model's $P(X_1)$ (i.e. `PROBS_1[START]`), then switches to the second-order
table from the second word onward.

In [10]:
def generate_second_order(probs=PROBS_2, max_len=20, rng=random):
    first = sample_next(START, PROBS_1, rng)
    tokens = [START, first]
    while tokens[-1] != END and len(tokens) < max_len:
        ctx = (tokens[-2], tokens[-1])
        dist = probs.get(ctx, {})
        if not dist:
            break
        words, weights = zip(*dist.items())
        tokens.append(rng.choices(words, weights=weights, k=1)[0])
    return " ".join(t for t in tokens if t not in (START, END))

random.seed(5)
for _ in range(5):
    print(generate_second_order())

the dog ran to the park
the dog sat on the mat
the dog sat on the rug
the dog sat on the mat
the cat ran to the park


## Question 11: how the second-order model differs from the first-order one

1. **Graph structure:** the first-order network is a simple chain, each node with one
   parent; the second-order network has each $X_t$ with *two* parents, $X_{t-2}$ and
   $X_{t-1}$.
2. **Conditional probability table:** the first-order table is indexed by a single previous
   word; the second-order table is indexed by a *pair* of previous words, so it is
   considerably larger (quadratic in vocabulary size rather than linear).
3. **Context available for prediction:** the second-order model sees two tokens of history
   instead of one, so it can in principle capture longer-range regularities (e.g.
   distinguishing "the cat sat" from "the dog sat" when predicting what follows "sat").
4. **Data needed:** because the table is indexed by pairs, far more training data is needed
   to observe each context more than a handful of times - with only six short sentences,
   many pairs here are seen only once or twice.

## Part XIII - Question 12: comparing the two models

In [11]:
def diversity(order, n=20, seed=11):
    rng = random.Random(seed)
    gen = generate_second_order if order == 2 else generate
    return {gen(rng=rng) for _ in range(n)}

n_params_1 = sum(len(d) for d in PROBS_1.values())
n_params_2 = sum(len(d) for d in PROBS_2.values())
n_zero_1 = sum(1 for w in VOCAB for d in [PROBS_1.get(w, {})] if len(d) < len(VOCAB))
contexts_2 = {(a, b) for toks in CORPUS for a, b in zip(toks, toks[1:])}
n_zero_2 = sum(1 for ctx in contexts_2 if ctx not in PROBS_2)

print(f"distinct parameters   : first-order={n_params_1}   second-order={n_params_2}")
print(f"zero-probability contexts (out of vocab, out of seen pairs): "
      f"first-order~{n_zero_1}   second-order~{n_zero_2}")

d1 = diversity(1)
d2 = diversity(2)
print(f"\ndistinct sentences out of 20 samples: first-order={len(d1)}   second-order={len(d2)}")
print("first-order examples: ", list(d1)[:5])
print("second-order examples:", list(d2)[:5])

distinct parameters   : first-order=17   second-order=18
zero-probability contexts (out of vocab, out of seen pairs): first-order~12   second-order~3

distinct sentences out of 20 samples: first-order=11   second-order=6
first-order examples:  ['the rug', 'the mat', 'the cat sat on the dog sat on the dog sat on the rug', 'the dog sat on the cat sat on the mat', 'the cat sat on the cat sat on the cat ran to the rug']
second-order examples: ['the dog sat on the rug', 'the cat sat on the rug', 'the cat sat on the mat', 'the cat ran to the park', 'the dog ran to the park']


The second-order model has noticeably more parameters (one distribution per *pair* of
preceding words rather than per single word), and correspondingly more contexts it has never
seen a transition from. Qualitatively, its generated sentences tend to stay closer to exact
training sentences, since with only two prior tokens' worth of context and this little data,
most contexts have only ever been followed by one specific word - there is less
freedom for the sampler to recombine words in new ways than the first-order model has.

## Question 12: why more context cuts both ways

Conditioning on more preceding tokens lets the model, in principle, capture dependencies a
shorter-context model cannot see (e.g. agreement or topic consistency over several words),
so predictions can become sharper and more specific. But the table size grows with the
number of *distinct contexts*, which grows roughly as (vocabulary size)^(context length):
with a fixed small corpus, almost every long context is seen at most once or twice, so the
estimated probabilities become unreliable (often collapsing to a single observed
continuation) - more context demands proportionally more data to estimate well.

## Part XIV/XV - from a hand-built table to a trained neural network

A modern autoregressive model still computes $P(X_t \mid X_1,\dots,X_{t-1})$, but instead of
a small hand-built table indexed by the last word or two, a neural network is trained
(by gradient descent, not counting) to produce that conditional distribution from a much
longer context window. The underlying generation loop - sample a token, append it, repeat -
is unchanged; what differs is purely *how* the conditional distribution is represented and
learned (a comparison table is in the handout: counts/fixed window vs. learned weights/large
learned context).

## Question 13: why specify behaviour before asking for code

Given a vague prompt ("write a language model"), an LLM is free to choose any
architecture at all - a pretrained transformer wrapper, a Markov chain, anything - so there
is nothing concrete to check the result against. Specifying "estimate
$P(X_t \mid X_{t-1})$ from transition counts, generate by sampling" first means:

- the **intended behaviour** is pinned down before any code exists;
- the **representation** (a CPT built from counts, not a neural net) is explicit, so the
  generated code can be checked against that choice line by line;
- the implementation can then be **validated** against the stated probabilistic invariant
  (every row sums to 1), not just against "does it run";
- **probabilistic invariants** like normalisation give an objective pass/fail test that
  plausible-looking but wrong code can fail;
- and the **implementation stays distinguishable from the model**: a bug in the Python code
  is a bug in the engineering, not evidence that the chain-rule idea itself is wrong.

## Question 14: what thinking in terms of a Bayesian network added

Three of the ways it helped, concretely:

- **A factorisation of the joint distribution.** Rather than one intractable
  $P(X_1,\dots,X_T)$, the chain-rule view turned this into a product of much simpler
  one-step conditionals, which is exactly what made counting-based estimation feasible here.
- **A principled method for generation.** Sampling $X_1$, then $X_2 \mid X_1$, then
  $X_3 \mid X_2$, and so on is not an ad-hoc trick; it is just forward sampling from the
  network, guaranteed to produce a sample from the joint distribution the network
  represents.
- **A way to understand the effect of increasing context.** Moving from the chain
  $X_{t-1}\to X_t$ to the collider $X_{t-2}, X_{t-1} \to X_t$ made the first/second-order
  comparison in Part XIII a question about graph structure and table size, rather than just
  which model's numbers happen to look bigger.

## Reflection on using the LLM

The LLM was most useful for producing the counting/CPT boilerplate quickly (Parts IV-V) and
for the mechanical trigram extension (Part XI), which would otherwise have been a repetitive
rewrite of the same bigram logic. One piece of generated code that needed correcting: an
early version of `generate` could loop forever if a sampled context was never followed by
`<END>` within the corpus (since `<END>` only has nonzero probability from certain
contexts) - fixed by adding the `max_len` cutoff used above, after noticing a test run that
never terminated. More generally, the normalisation check in Part VII is the kind of test
that would catch a subtle counting bug (e.g. double-counting a bigram, or forgetting the
`<END>` token) that would otherwise be invisible just from looking at generated sentences
that happen to look reasonable.